# E-Commerce Sales & Customer Insights

This notebook explores the UCI Online Retail transactions. Download `Online Retail.xlsx` from https://archive.ics.uci.edu/dataset/352/online+retail and put it in `data/raw/`. Run cells in order.

The goal is to create reproducible analysis, not to assume conclusions before checking the data.

## 1. Import libraries and load the data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
DATA_FILE = ROOT / 'data' / 'raw' / 'Online Retail.xlsx'
if not DATA_FILE.exists():
    raise FileNotFoundError(f'Download Online Retail.xlsx from the UCI page and save it to {DATA_FILE}')

df = pd.read_excel(DATA_FILE)
df.columns = [str(c).strip() for c in df.columns]
print('Rows and columns:', df.shape)
display(df.head())

## 2. Inspect data quality

Review column types, missing values, and exact duplicates before deciding how to handle them.

In [ ]:
display(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
print('Exact duplicate rows:', int(df.duplicated().sum()))

## 3. Clean transactions

Cancellation invoices start with `C` in this dataset. Keep flagged rows separately, and use only positive-quantity, positive-price transactions to analyze gross sales. Missing customer IDs remain in sales totals but cannot be used for customer-level metrics.

In [ ]:
required = {'InvoiceNo','StockCode','Description','Quantity','InvoiceDate','UnitPrice','CustomerID','Country'}
missing_cols = required - set(df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

data = df.copy()
data['InvoiceNo'] = data['InvoiceNo'].astype('string').str.strip()
data['StockCode'] = data['StockCode'].astype('string').str.strip()
data['Description'] = data['Description'].astype('string').str.strip()
data['Country'] = data['Country'].astype('string').str.strip()
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], errors='coerce')
data['Quantity'] = pd.to_numeric(data['Quantity'], errors='coerce')
data['UnitPrice'] = pd.to_numeric(data['UnitPrice'], errors='coerce')
data['CustomerID'] = pd.to_numeric(data['CustomerID'], errors='coerce').astype('Int64')
duplicates_removed = int(data.duplicated().sum())
data = data.drop_duplicates().dropna(subset=['InvoiceNo','StockCode','InvoiceDate','Quantity','UnitPrice','Country']).copy()
data['IsCancellationInvoice'] = data['InvoiceNo'].str.upper().str.startswith('C', na=False)
data['IsNegativeQuantity'] = data['Quantity'] < 0
data['IsReturnOrCancellationLike'] = data['IsCancellationInvoice'] | data['IsNegativeQuantity']
data['LineRevenue'] = data['Quantity'] * data['UnitPrice']
flagged = data.loc[data['IsReturnOrCancellationLike']].copy()
sales = data.loc[(~data['IsReturnOrCancellationLike']) & (data['Quantity'] > 0) & (data['UnitPrice'] > 0)].copy()
sales['Description'] = sales['Description'].fillna('Unknown product')
sales['InvoiceMonth'] = sales['InvoiceDate'].dt.to_period('M').astype(str)
print(f'Exact duplicates removed: {duplicates_removed:,}')
print(f'Clean positive-sales rows: {len(sales):,}')
print(f'Cancellation/return-like rows: {len(flagged):,}')
display(sales.head())

## 4. Calculate key metrics

In [ ]:
total_revenue = sales['LineRevenue'].sum()
total_orders = sales['InvoiceNo'].nunique()
identified_customers = sales['CustomerID'].nunique()
aov = total_revenue / total_orders if total_orders else np.nan
kpis = pd.DataFrame({'Metric':['Gross sales revenue','Distinct invoices','Identified customers','Average order value'],
                     'Value':[total_revenue,total_orders,identified_customers,aov]})
display(kpis)

## 5. Monthly trend

In [ ]:
monthly = sales.groupby('InvoiceMonth', as_index=False).agg(Revenue=('LineRevenue','sum'), Orders=('InvoiceNo','nunique'))
display(monthly)
plt.figure(figsize=(11,4))
plt.plot(monthly['InvoiceMonth'], monthly['Revenue'], marker='o')
plt.title('Monthly Sales Revenue')
plt.xlabel('Month')
plt.ylabel('Revenue (dataset currency)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Product and country performance

In [ ]:
top_products = (sales.groupby(['StockCode','Description'], as_index=False)
               .agg(Revenue=('LineRevenue','sum'), Units_Sold=('Quantity','sum'))
               .sort_values('Revenue', ascending=False).head(10))
display(top_products)

country_sales = (sales.groupby('Country', as_index=False)
                 .agg(Revenue=('LineRevenue','sum'), Orders=('InvoiceNo','nunique'))
                 .sort_values('Revenue', ascending=False))
display(country_sales.head(10))

## 7. Customer purchase behaviour

RFM-style metrics use the latest transaction date in the dataset as a snapshot date, not today's date.

In [ ]:
customer_sales = sales.dropna(subset=['CustomerID']).copy()
snapshot_date = customer_sales['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
customer_summary = customer_sales.groupby('CustomerID', as_index=False).agg(
    Last_Order_Date=('InvoiceDate','max'), Frequency=('InvoiceNo','nunique'),
    Monetary_Value=('LineRevenue','sum'), Total_Units=('Quantity','sum'))
customer_summary['Recency_Days'] = (snapshot_date - customer_summary['Last_Order_Date'].dt.normalize()).dt.days
customer_summary = customer_summary[['CustomerID','Recency_Days','Frequency','Monetary_Value','Total_Units']]
display(customer_summary.sort_values('Monetary_Value', ascending=False).head(10))
repeat_pct = (customer_summary['Frequency'].gt(1).mean() * 100) if not customer_summary.empty else np.nan
print(f'Repeat customers among customers with IDs: {repeat_pct:.2f}%')

## 8. Export outputs for SQL and Power BI

The CSV files are generated locally and ignored by Git because they can be recreated. Add selected screenshots to the repository after reviewing the results.

In [ ]:
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
export_cols = ['InvoiceNo','StockCode','Description','Quantity','InvoiceDate','UnitPrice','CustomerID','Country','LineRevenue','InvoiceMonth']
sales[export_cols].to_csv(OUT / 'sales_clean.csv', index=False)
monthly.to_csv(OUT / 'monthly_sales.csv', index=False)
top_products.to_csv(OUT / 'top_products.csv', index=False)
country_sales.to_csv(OUT / 'country_sales.csv', index=False)
customer_summary.to_csv(OUT / 'customer_summary.csv', index=False)
flagged.assign(InvoiceMonth=flagged['InvoiceDate'].dt.to_period('M').astype(str)).groupby('InvoiceMonth', as_index=False).agg(Flagged_Rows=('InvoiceNo','size'), Flagged_Invoices=('InvoiceNo','nunique'), Signed_Line_Value=('LineRevenue','sum')).to_csv(OUT / 'return_cancellation_summary.csv', index=False)
print('Saved analysis outputs to:', OUT)

## 9. Write evidence-based conclusions

After running all cells, write 3–5 conclusions supported by the results. For example, identify the strongest sales month, the top product and the countries with high revenue. Do not use generic conclusions unless your own output supports them. Also mention limitations: one year of historical transactions, missing customer IDs, and cancellation/return flags being an approximation rather than proof of completed refunds.